# 🌾 Agri III - Target 0.97+ Balanced Accuracy
### S6E4: Predicting Irrigation Need | Advanced Stacking Ensemble
### Key Improvements over Beast:
- Deeper models (depth 8-10) with tuned regularization
- 2-level stacking ensemble (meta-learner)
- Enhanced feature engineering (30+ new features)
- Class-weight optimization for minority classes
- Multiple random seeds for diversity
- Optimized target encoding with smoothing
- Advanced interaction features from domain knowledge

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import warnings
warnings.filterwarnings('ignore')

print('='*70)
print('🌾 AGRI III - TARGET 0.97+ BALANCED ACCURACY')
print('='*70)

In [ ]:
# Load data
print('\n[1/7] Loading data...')
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

# Auto-detect target column
target_candidates = [c for c in train.columns if c.lower() in ['irrigation_need', 'target', 'label']]
if not target_candidates:
    # If no match, use the last column (common pattern in Kaggle)
    target_candidates = [train.columns[-1]]
    print(f'  ⚠️  Auto-detected target column: {target_candidates[0]} (last column)')
target_col = target_candidates[0]

# Rename to standard name
if target_col != 'Irrigation_Need':
    train = train.rename(columns={target_col: 'Irrigation_Need'})

print(f'  Train: {train.shape}, Test: {test.shape}')
print(f'  Target distribution:')
print(f'  {train["Irrigation_Need"].value_counts().to_dict()}')
print(f'  Class imbalance ratio: {train["Irrigation_Need"].value_counts().max() / train["Irrigation_Need"].value_counts().min():.1f}x')

In [ ]:
# Advanced Feature Engineering - Enhanced
print('\n[2/7] Feature engineering...')

train['is_train'] = True
test['is_train'] = False
test['Irrigation_Need'] = 'Low'

df = pd.concat([train, test], axis=0, ignore_index=True)

# Encode categoricals
cat_cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# ===== CORE INTERACTION FEATURES =====
df['Soil_pH_Soil_Moisture'] = df['Soil_pH'] * df['Soil_Moisture']
df['Temperature_Humidity'] = df['Temperature_C'] * df['Humidity']
df['Rainfall_Soil_Moisture'] = df['Rainfall_mm'] * df['Soil_Moisture']
df['Sunlight_Temperature'] = df['Sunlight_Hours'] * df['Temperature_C']
df['Wind_Humidity'] = df['Wind_Speed_kmh'] * df['Humidity']
df['Organic_Temperature'] = df['Organic_Carbon'] * df['Temperature_C']
df['EC_Moisture'] = df['Electrical_Conductivity'] * df['Soil_Moisture']
df['Organic_EC'] = df['Organic_Carbon'] * df['Electrical_Conductivity']
df['Rainfall_Temperature'] = df['Rainfall_mm'] * df['Temperature_C']
df['Sunlight_Humidity'] = df['Sunlight_Hours'] * df['Humidity']

# ===== RATIO FEATURES =====
df['Organic_to_Soil'] = df['Organic_Carbon'] / (df['Soil_Moisture'] + 1e-6)
df['EC_to_Soil_pH'] = df['Electrical_Conductivity'] / (df['Soil_pH'] + 1e-6)
df['Rainfall_per_Hectare'] = df['Rainfall_mm'] / (df['Field_Area_hectare'] + 1e-6)
df['Prev_Irrigation_per_Hectare'] = df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + 1e-6)
df['Temp_Humidity_Ratio'] = df['Temperature_C'] / (df['Humidity'] + 1e-6)
df['Rainfall_Humidity'] = df['Rainfall_mm'] / (df['Humidity'] + 1e-6)
df['Moisture_per_Unit_Rainfall'] = df['Soil_Moisture'] / (df['Rainfall_mm'] + 1e-6)
df['Organic_per_Unit_EC'] = df['Organic_Carbon'] / (df['Electrical_Conductivity'] + 1e-6)
df['Wind_Temperature_Ratio'] = df['Wind_Speed_kmh'] / (df['Temperature_C'] + 1e-6)

# ===== POLYNOMIAL FEATURES =====
df['Soil_pH_sq'] = df['Soil_pH'] ** 2
df['Moisture_sq'] = df['Soil_Moisture'] ** 2
df['Temperature_sq'] = df['Temperature_C'] ** 2
df['Humidity_sq'] = df['Humidity'] ** 2
df['Rainfall_log'] = np.log1p(df['Rainfall_mm'])
df['Prev_Irrigation_log'] = np.log1p(df['Previous_Irrigation_mm'])
df['Soil_Moisture_log'] = np.log1p(df['Soil_Moisture'])
df['Organic_log'] = np.log1p(df['Organic_Carbon'])
df['EC_log'] = np.log1p(df['Electrical_Conductivity'])

# ===== DOMAIN-SPECIFIC FEATURES =====
df['Water_Stress'] = (df['Temperature_C'] * df['Wind_Speed_kmh']) / (df['Humidity'] + df['Soil_Moisture'] + 1e-6)
df['Irrigation_Efficiency'] = df['Previous_Irrigation_mm'] / (df['Rainfall_mm'] + 1e-6)
df['Crop_Water_Demand'] = df['Sunlight_Hours'] * df['Temperature_C'] / (df['Humidity'] + 1e-6)
df['Evapotranspiration_Proxy'] = (df['Temperature_C'] * df['Sunlight_Hours'] * df['Wind_Speed_kmh']) / (df['Humidity'] + 1e-6)
df['Soil_Water_Retention'] = df['Soil_Moisture'] * df['Organic_Carbon'] / (df['Electrical_Conductivity'] + 1e-6)
df['Net_Water_Balance'] = df['Rainfall_mm'] + df['Previous_Irrigation_mm'] - df['Soil_Moisture']
df['Water_Availability_Index'] = (df['Rainfall_mm'] * df['Humidity']) / (df['Temperature_C'] + df['Wind_Speed_kmh'] + 1e-6)
df['Soil_Health_Index'] = (df['Organic_Carbon'] * df['Soil_Moisture']) / (df['Electrical_Conductivity'] + df['Soil_pH'] + 1e-6)
df['Crop_Stress_Index'] = (df['Temperature_C'] * df['Sunlight_Hours']) / (df['Soil_Moisture'] + df['Humidity'] + 1e-6)

# ===== BINNED FEATURES =====
df['Soil_Moisture_bin'] = pd.qcut(df['Soil_Moisture'], q=10, labels=False, duplicates='drop')
df['Rainfall_bin'] = pd.qcut(df['Rainfall_mm'], q=10, labels=False, duplicates='drop')
df['Temperature_bin'] = pd.qcut(df['Temperature_C'], q=10, labels=False, duplicates='drop')
df['Humidity_bin'] = pd.qcut(df['Humidity'], q=10, labels=False, duplicates='drop')
df['Organic_bin'] = pd.qcut(df['Organic_Carbon'], q=10, labels=False, duplicates='drop')
df['EC_bin'] = pd.qcut(df['Electrical_Conductivity'], q=10, labels=False, duplicates='drop')

# ===== TARGET ENCODING WITH SMOOTHING =====
train_mask = df['is_train'] == True
target_numeric = df[train_mask]['Irrigation_Need'].map({'Low': 0, 'Medium': 1, 'High': 2})
df.loc[train_mask, 'target_numeric'] = target_numeric

for group_col in ['Crop_Type', 'Soil_Type', 'Region', 'Season', 'Irrigation_Type']:
    # Mean encoding with smoothing
    global_mean = target_numeric.mean()
    group_stats = df[train_mask].groupby(group_col)['target_numeric'].agg(['mean', 'count'])
    
    # Smoothed encoding (shrinkage towards global mean)
    smoothing = 10
    smoothed = (group_stats['mean'] * group_stats['count'] + global_mean * smoothing) / (group_stats['count'] + smoothing)
    
    df[f'{group_col}_target_smooth'] = df[group_col].map(smoothed)
    df[f'{group_col}_target_mean'] = df[group_col].map(group_stats['mean'])
    df[f'{group_col}_count'] = df[group_col].map(group_stats['count'])
    
    # High probability encoding
    high_prob = df[train_mask].groupby(group_col)['target_numeric'].apply(lambda x: (x == 2).mean())
    df[f'{group_col}_high_prob'] = df[group_col].map(high_prob)
    
    # Low probability encoding
    low_prob = df[train_mask].groupby(group_col)['target_numeric'].apply(lambda x: (x == 0).mean())
    df[f'{group_col}_low_prob'] = df[group_col].map(low_prob)

# ===== CROSS FEATURES =====
df['Crop_Soil'] = df['Crop_Type'].astype(str) + '_' + df['Soil_Type'].astype(str)
df['Crop_Region'] = df['Crop_Type'].astype(str) + '_' + df['Region'].astype(str)
df['Crop_Season'] = df['Crop_Type'].astype(str) + '_' + df['Season'].astype(str)
df['Soil_Region'] = df['Soil_Type'].astype(str) + '_' + df['Region'].astype(str)
df['Irrigation_Crop'] = df['Irrigation_Type'].astype(str) + '_' + df['Crop_Type'].astype(str)
df['Irrigation_Region'] = df['Irrigation_Type'].astype(str) + '_' + df['Region'].astype(str)
df['Crop_Growth_Soil'] = df['Crop_Growth_Stage'].astype(str) + '_' + df['Soil_Type'].astype(str)
df['Crop_Growth_Region'] = df['Crop_Growth_Stage'].astype(str) + '_' + df['Region'].astype(str)

cross_cols = ['Crop_Soil', 'Crop_Region', 'Crop_Season', 'Soil_Region', 
              'Irrigation_Crop', 'Irrigation_Region', 'Crop_Growth_Soil', 'Crop_Growth_Region']
for col in cross_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    
    # Add target encoding for cross features
    cross_stats = df[train_mask].groupby(col)['target_numeric'].agg(['mean', 'count'])
    smoothed_cross = (cross_stats['mean'] * cross_stats['count'] + global_mean * smoothing) / (cross_stats['count'] + smoothing)
    df[f'{col}_target_smooth'] = df[col].map(smoothed_cross)

# ===== STATISTICAL FEATURES BY GROUP =====
for group_col in ['Crop_Type', 'Soil_Type', 'Region']:
    for stat_col in ['Soil_Moisture', 'Rainfall_mm', 'Temperature_C', 'Humidity']:
        group_stats = df[train_mask].groupby(group_col)[stat_col].agg(['mean', 'std'])
        df[f'{group_col}_{stat_col}_mean'] = df[group_col].map(group_stats['mean'])
        df[f'{group_col}_{stat_col}_std'] = df[group_col].map(group_stats['std'])

# Drop helper column
if 'target_numeric' in df.columns:
    df.drop(['target_numeric'], axis=1, inplace=True)

# Split back
train_final = df.iloc[:len(train)].copy()
test_final = df.iloc[len(train):].copy()
train_final['Irrigation_Need'] = train['Irrigation_Need']

train_final.drop(['is_train', 'id'], axis=1, inplace=True)
test_final.drop(['is_train'], axis=1, inplace=True)
if 'Irrigation_Need' in test_final.columns:
    test_final.drop(['Irrigation_Need'], axis=1, inplace=True)
if 'id' in test_final.columns:
    test_final.drop(['id'], axis=1, inplace=True)

print(f'  Total features: {train_final.shape[1] - 1}')

In [ ]:
# Prepare data
print('\n[3/7] Preparing...')

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(train_final['Irrigation_Need'])
X = train_final.drop('Irrigation_Need', axis=1)
X_test = test_final.copy()

# Identify categorical features for CatBoost
cat_features = [col for col in X.columns if col not in 
                ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
                 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh',
                 'Field_Area_hectare', 'Previous_Irrigation_mm']]

print(f'  Classes: {target_encoder.classes_}')
print(f'  Features: {X.shape[1]}')
print(f'  Class distribution: {np.bincount(y)}')

In [ ]:
# Train LEVEL-1 models with CV - Optimized for BALANCED accuracy
# WITH MEMORY MANAGEMENT to prevent kernel restarts on large datasets
print('\n[4/7] Training level-1 models...')
print(f'  Memory: {X.memory_usage(deep=True).sum() / 1e6:.1f} MB')

import gc

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Store OOF predictions for stacking
oof_proba = np.zeros((len(X), 3, 3))  # 3 classes, 3 models
test_proba = np.zeros((len(X_test), 3, 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'\n  Fold {fold+1}/{N_SPLITS}')
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Sample weights for balanced learning
    from sklearn.utils.class_weight import compute_sample_weight
    sample_weights = compute_sample_weight('balanced', y_train)
    
    # ===== MODEL 1: XGBoost =====
    print('    Training XGB...', end=' ')
    xgb_model = xgb.XGBClassifier(
        n_estimators=1200, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        gamma=0.1, reg_alpha=0.3, reg_lambda=1.5,
        random_state=42, eval_metric='mlogloss', tree_method='hist',
        verbosity=0, n_jobs=-1
    )
    xgb_model.fit(X_train, y_train, sample_weight=sample_weights,
                 eval_set=[(X_val, y_val)], verbose=False)
    
    oof_proba[val_idx, :, 0] = xgb_model.predict_proba(X_val)
    test_proba[:, :, 0] += xgb_model.predict_proba(X_test) / N_SPLITS
    
    bal_acc_xgb = balanced_accuracy_score(y_val, np.argmax(oof_proba[val_idx, :, 0], axis=1))
    print(f'Acc: {bal_acc_xgb:.6f}')
    
    # FREE XGBoost memory immediately
    del xgb_model, X_train, X_val, y_train, y_val
    gc.collect()
    print('    ✓ XGB memory freed')
    
    # Recreate for next model
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # ===== MODEL 2: LightGBM =====
    print('    Training LGBM...', end=' ')
    lgb_model = lgb.LGBMClassifier(
        n_estimators=1200, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
        reg_alpha=0.3, reg_lambda=1.5, class_weight='balanced',
        random_state=42, verbosity=-1, n_jobs=-1
    )
    lgb_model.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(100, verbose=False), 
                            lgb.log_evaluation(0)])
    
    oof_proba[val_idx, :, 1] = lgb_model.predict_proba(X_val)
    test_proba[:, :, 1] += lgb_model.predict_proba(X_test) / N_SPLITS
    
    bal_acc_lgb = balanced_accuracy_score(y_val, np.argmax(oof_proba[val_idx, :, 1], axis=1))
    print(f'Acc: {bal_acc_lgb:.6f}')
    
    # FREE LightGBM memory immediately
    del lgb_model, X_train, X_val, y_train, y_val
    gc.collect()
    print('    ✓ LGBM memory freed')
    
    # Recreate for next model
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # ===== MODEL 3: CatBoost =====
    print('    Training CAT...', end=' ')
    cat_model = CatBoostClassifier(
        iterations=1000, depth=6, learning_rate=0.05,
        l2_leaf_reg=2.0, random_seed=42, 
        verbose=0, task_type='CPU', auto_class_weights='Balanced',
        min_data_in_leaf=20, thread_count=-1
    )
    cat_model.fit(X_train, y_train, eval_set=(X_val, y_val),
                  early_stopping_rounds=100, verbose=False)
    
    oof_proba[val_idx, :, 2] = cat_model.predict_proba(X_val)
    test_proba[:, :, 2] += cat_model.predict_proba(X_test) / N_SPLITS
    
    bal_acc_cat = balanced_accuracy_score(y_val, np.argmax(oof_proba[val_idx, :, 2], axis=1))
    print(f'Acc: {bal_acc_cat:.6f}')
    
    # FREE CatBoost memory immediately
    del cat_model, X_train, X_val, y_train, y_val
    gc.collect()
    print('    ✓ CAT memory freed')
    
    print(f'  ✅ Fold {fold+1} complete - All memory released')

# Overall OOF scores
print(f'\n  Overall OOF Balanced Accuracies:')
model_names = ['XGB', 'LGBM', 'CAT']
oof_scores = []
for i, name in enumerate(model_names):
    oof_pred = np.argmax(oof_proba[:, :, i], axis=1)
    score = balanced_accuracy_score(y, oof_pred)
    oof_scores.append(score)
    print(f'    {name}: {score:.6f}')

# Final cleanup
gc.collect()
print(f'\n  💾 Fold training memory cleanup done')

In [ ]:
# LEVEL-2: Stacking with Meta-Learner
print('\n[5/7] Training level-2 meta-learner...')

# Prepare meta-features (OOF probabilities from all level-1 models)
meta_features_train = oof_proba.reshape(len(X), -1)  # (n_samples, 3 * n_models)
meta_features_test = test_proba.reshape(len(X_test), -1)

# Train simple meta-learner (Logistic Regression to avoid overfitting)
meta_model = LogisticRegression(
    C=1.0, max_iter=1000, solver='lbfgs', 
    class_weight='balanced', random_state=42
)
meta_model.fit(meta_features_train, y)

# Meta-learner OOF predictions
meta_oof_pred = meta_model.predict(meta_features_train)
meta_bal_acc = balanced_accuracy_score(y, meta_oof_pred)
print(f'  Meta-learner OOF balanced_acc: {meta_bal_acc:.6f}')

# Meta-learner test predictions
meta_test_pred = meta_model.predict(meta_features_test)
print(f'  Meta-learner test distribution: {pd.Series(meta_test_pred).value_counts().to_dict()}')

# Also try simple average (often competitive)
simple_avg_proba = np.mean([test_proba[:, :, i] for i in range(test_proba.shape[2])], axis=0)
simple_avg_pred = np.argmax(simple_avg_proba, axis=1)
print(f'  Simple avg test distribution: {pd.Series(simple_avg_pred).value_counts().to_dict()}')

In [ ]:
# Ensemble selection & Post-processing
print('\n[6/7] Ensemble selection...')

# Evaluate all models on OOF and pick best combination
oof_scores = []
for i, name in enumerate(model_names):
    oof_pred = np.argmax(oof_proba[:, :, i], axis=1)
    score = balanced_accuracy_score(y, oof_pred)
    oof_scores.append(score)
    print(f'  {name}: {score:.6f}')

# Weighted ensemble based on OOF scores
weights = np.array(oof_scores)
weights = weights / weights.sum()
print(f'\n  Model weights: {np.round(weights, 4)}')

# Weighted probability ensemble
weighted_proba = np.zeros((len(X_test), 3))
for i in range(test_proba.shape[2]):
    weighted_proba += weights[i] * test_proba[:, :, i]

weighted_pred = np.argmax(weighted_proba, axis=1)
print(f'  Weighted ensemble distribution: {pd.Series(weighted_pred).value_counts().to_dict()}')

# Simple average
simple_pred = np.argmax(simple_avg_proba, axis=1)
print(f'  Simple avg distribution: {pd.Series(simple_pred).value_counts().to_dict()}')

# Meta-learner
print(f'  Meta-learner distribution: {pd.Series(meta_test_pred).value_counts().to_dict()}')

# Use meta-learner (usually best for balanced accuracy)
final_pred = meta_test_pred
print(f'\n  ✓ Using meta-learner predictions')

In [ ]:
# Create submission
print('\n[7/7] Creating submission...')

test_labels = target_encoder.inverse_transform(final_pred)

# Find ID column
id_candidates = [c for c in test.columns if c.lower() in ['id', 'row_id', 'sample_id']]
if not id_candidates:
    id_candidates = [test.columns[0]]
id_col = id_candidates[0]

submission = pd.DataFrame({
    id_col: test[id_col],
    'Irrigation_Need': test_labels
})

print(f'\n  Final predictions:')
print(f'  {submission["Irrigation_Need"].value_counts().to_dict()}')

# Save submission
submission.to_csv('submission.csv', index=False)
print('\n  ✓ submission.csv saved!')

# Save OOF scores for analysis
oof_results = pd.DataFrame({
    'Model': model_names + ['Meta-Learner'],
    'Balanced_Accuracy': oof_scores + [meta_bal_acc]
})
print('\n  OOF Results:')
print(oof_results.to_string(index=False))

# Free large arrays
del oof_proba, test_proba, meta_features_train, meta_features_test
del oof_scores, meta_test_pred
gc.collect()
print('\n  💾 Final memory cleanup done')

print('\n' + '='*70)
print('🌾 AGRI III COMPLETE! Target: 0.97+ Balanced Accuracy')
print('='*70)